# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and analyzing the FAIR² dataset ([Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

- **Title**: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells
- **Description**: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes various fields such as accession, description, coverage percentage, peptide counts, and molecular weight (MW) alongside calculated parameters such as pI and normalized abundances across different samples.
- **Croissant Schema URL**: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed!pip install mlcroissant

## 1. Data Loading

Load Croissant dataset metadata and inspect high-level information.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)

# Inspect and print metadata: name and description
meta = dataset.metadata
print(f"Dataset loaded: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Published: {meta.datePublished}")
print(f"Keywords: {getattr(meta, 'keywords', None)}")

## 2. Data Overview

We'll review the record sets, fields, and columns available in the dataset schema. Each entity is referenced by its `@id`, as required for Croissant datasets.

In [ ]:
# List available record sets, their @ids and fields
record_sets = dataset.metadata.record_sets

print(f"Total record sets: {len(record_sets)}\n")
for i, rs in enumerate(record_sets):
    print(f"[{i}] Record Set Name: {rs.name if hasattr(rs, 'name') else ''}")
    print(f"    @id: {rs.id}")
    if hasattr(rs, 'fields'):
        print(f"    Fields:")
        for field in rs.fields:
            print(f"        - {field.name if hasattr(field, 'name') else ''} (@id: {field.id})")
    print("")

## 3. Data Extraction

We'll load data from one or more record sets into Pandas DataFrames.

First, let's collect all record set `@id`s. Then we'll demonstrate how to extract records from a selected record set using its `@id`.

In [ ]:
# Collect all record set @id values
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Available Record Set @ids:")
for rid in record_sets_ids:
    print("  ", rid)

# Load all record sets into DataFrames
dataframes = {}
for rs in dataset.metadata.record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)

# Pick the first record set for demonstration
selected_rs_id = record_sets_ids[0] if len(record_sets_ids) > 0 else None
if selected_rs_id is not None and not dataframes[selected_rs_id].empty:
    print(f"Columns for record set {selected_rs_id}:")
    print(dataframes[selected_rs_id].columns.tolist())
    dataframes[selected_rs_id].head()
else:
    print("No data available in the selected record set.")

## 4. Exploratory Data Analysis (EDA)

Let's perform standard exploratory data analysis tasks on a single record set, referencing all columns/fields by `@id` as needed.

Assuming one of the fields is numeric (for example, 'mw' for molecular weight, commonly present in proteomics datasets), you would use its field `@id`. If not, select any actual numeric `@id` as found in the earlier outputs.

In [ ]:
# Replace '<numeric_field_id>' and '<group_field_id>' with valid field @ids from the above overview.
if selected_rs_id is not None and not dataframes[selected_rs_id].empty:
    df = dataframes[selected_rs_id].copy()
    print(f"Examining DataFrame from record set @id: {selected_rs_id}")

    # Guess a numeric field if possible
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    print("Numeric fields:", numeric_fields)
    numeric_field = numeric_fields[0] if len(numeric_fields) > 0 else None
    if numeric_field:
        # Filtering based on a threshold
        threshold = df[numeric_field].mean() if df[numeric_field].mean() is not None else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records where '{numeric_field}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field if available
        group_fields = [c for c in df.columns if df[c].dtype == object or df[c].dtype.name == 'category']
        group_field = group_fields[0] if len(group_fields) > 0 else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())
        else:
            print("\nNo suitable group field found for grouping.")
    else:
        print("\nNo numeric field available for EDA.")
else:
    print("No non-empty record sets to perform EDA.")

## 5. Visualization

Let's visualize the distribution of a selected numeric field in the selected record set. We'll create a histogram for demonstration, using field `@id` as the x-axis.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id is not None and not dataframes[selected_rs_id].empty and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[selected_rs_id][numeric_field].dropna(), bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of '{numeric_field}' in record set {selected_rs_id}")
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

- The dataset provides rich, structured mass spectrometry protein quantification records using the Croissant and FAIR² schema standards.
- Key entities and fields were identified through their Croissant `@id` and successfully loaded, explored, and visualized.
- You can now proceed to further, application-specific analyses (e.g., differential abundance testing, clustering, or data integration) using the processed and annotated DataFrames.

#### Tips:
- Always reference Croissant schema entities by their `@id` for reproducibility.
- Use the `mlcroissant` library to ensure future-proof data access and compliance with the FAIR principles.